
# Benchmark des modèles CityTaste

Ce notebook sert à comparer **3 modèles** sur le **même assistant CityTaste** via l’endpoint API `POST /api/chat`.

## Objectif
Évaluer chaque modèle sur :
- la **pertinence métier** de la réponse,
- la **bonne langue** de sortie,
- l’**absence de phrases prototype**,
- la **latence**,
- puis ajouter une **évaluation manuelle** pour départager deux modèles proches.

## Modèles à comparer
Exemples :
- modèle actuel `llama_cpp`
- `gemma3:4b`
- `qwen3:4b`

## Principe
1. Lancer l’API CityTaste avec un modèle.
2. Exécuter ce notebook.
3. Sauvegarder les résultats.
4. Changer de modèle.
5. Refaire exactement les mêmes tests.


In [31]:
import time
import re
from typing import Dict, Any, List

import pandas as pd
import requests


## Configuration

Change `MODEL_NAME` à chaque campagne de test.

Exemples :
- `llama_cpp_current`
- `gemma3_4b`
- `qwen3_4b`


In [ ]:
API_URL = "http://127.0.0.1:8000/api/chat"

MODEL_NAME = "gemma3:4b"

REQUEST_TIMEOUT = 600

## Jeu de questions de test


L’idée est de couvrir :
- recherche de lieux,
- FAQ site,
- langue française et anglaise,
- suivi conversationnel simple,
- cas à risque pour les phrases prototype.


In [42]:
TEST_CASES = [
    {
        "id": 1,
        "question": "Trouve-moi un restaurant italien proche du centre-ville d’Ottawa.",
        "context": None,
        "expected_lang": "fr",
        "must_include_any": ["restaurant", "ital", "centre", "ottawa"],
        "must_not_include": [
            "voici la réponse finale",
            "voici la reponse finale",
            "je vais chercher",
            "j'ai cherché",
            "consigne finale",
        ],
    },
    {
        "id": 2,
        "question": "Je cherche un restaurant chinois près de St-Laurent.",
        "context": None,
        "expected_lang": "fr",
        "must_include_any": ["restaurant", "chinois", "chinese", "st laurent", "st-laurent"],
        "must_not_include": [
            "voici la réponse finale",
            "je comprends que l'utilisateur",
            "je vais utiliser",
        ],
    },
    {
        "id": 3,
        "question": "Je veux un hôtel près du centre-ville.",
        "context": None,
        "expected_lang": "fr",
        "must_include_any": ["hôtel", "hotel", "centre", "ottawa"],
        "must_not_include": [
            "voici la réponse finale",
            "consigne finale",
            "je vais chercher",
        ],
    },
    {
        "id": 4,
        "question": "Find an Italian restaurant near downtown Ottawa.",
        "context": None,
        "expected_lang": "en",
        "must_include_any": ["restaurant", "italian", "downtown", "ottawa"],
        "must_not_include": [
            "voici la réponse finale",
            "je comprends que",
            "consigne finale",
        ],
    },
    {
        "id": 5,
        "question": "Find a hotel near downtown.",
        "context": None,
        "expected_lang": "en",
        "must_include_any": ["hotel", "downtown", "ottawa", "accommodation"],
        "must_not_include": [
            "here is the final answer",
            "i understand that the user",
            "consigne finale",
        ],
    },
    {
        "id": 6,
        "question": "How do I use filters in CityTaste?",
        "context": None,
        "expected_lang": "en",
        "must_include_any": ["filter", "results", "cuisine", "distance"],
        "must_not_include": [
            "here is the final answer",
            "consigne finale",
            "i will search",
        ],
    },
    {
        "id": 7,
        "question": "Comment utiliser les filtres dans CityTaste ?",
        "context": None,
        "expected_lang": "fr",
        "must_include_any": ["filtres", "résultats", "cuisine", "distance"],
        "must_not_include": [
            "voici la réponse finale",
            "consigne finale",
            "je vais utiliser",
        ],
    },
    {
        "id": 8,
        "question": "CityTaste permet-il de faire une réservation ?",
        "context": None,
        "expected_lang": "fr",
        "must_include_any": ["réservation", "site web", "réserver", "coordonnées"],
        "must_not_include": [
            "voici la réponse finale",
            "je vais chercher",
            "consigne finale",
        ],
    },
    {
        "id": 9,
        "question": "How are the results ranked?",
        "context": None,
        "expected_lang": "en",
        "must_include_any": ["relevance", "distance", "rating", "results"],
        "must_not_include": [
            "here is the final answer",
            "consigne finale",
            "i will search",
        ],
    },
    {
        "id": 10,
        "question": "Comment activer ma position ?",
        "context": None,
        "expected_lang": "fr",
        "must_include_any": ["position", "localisation", "navigateur", "autorisation"],
        "must_not_include": [
            "voici la réponse finale",
            "consigne finale",
            "je vais utiliser",
        ],
    },
    {
        "id": 11,
        "question": "Why is this place recommended?",
        "context": {
            "selected_place": {
                "name": "Paninaro",
                "place_type": "restaurant",
                "cuisine": "italian",
                "address": "82 O'Connor Street, Ottawa",
                "google_rating": 4.6,
                "google_user_rating_count": 400,
                "dist_to_center_km": 0.14,
            }
        },
        "expected_lang": "en",
        "must_include_any": ["recommended", "restaurant", "italian", "rating", "distance"],
        "must_not_include": [
            "here is the final answer",
            "i understand that the user",
            "consigne finale",
        ],
    },
    {
        "id": 12,
        "question": "Quelle est l’adresse de ce lieu ?",
        "context": {
            "selected_place": {
                "name": "Fragrant Noodles",
                "place_type": "restaurant",
                "cuisine": "chinese",
                "address": "Rideau Street, Ottawa",
            }
        },
        "expected_lang": "fr",
        "must_include_any": ["adresse", "rideau", "ottawa"],
        "must_not_include": [
            "voici la réponse finale",
            "je suis là pour t'aider",
            "consigne finale",
        ],
    },
]


## Fonctions utilitaires d’évaluation

Le score automatique proposé est sur **100** :
- **35%** pertinence du contenu
- **25%** bonne langue
- **25%** absence de phrases prototype
- **15%** rapidité


In [43]:
EN_HINTS = {
    "the", "and", "how", "find", "restaurant", "hotel", "filters", "results",
    "location", "distance", "rating", "booking", "near", "around"
}

FR_HINTS = {
    "le", "la", "les", "comment", "trouve", "restaurant", "hôtel", "filtres",
    "résultats", "localisation", "distance", "note", "réservation", "près", "zone"
}

def normalize_text(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def detect_lang_simple(text: str) -> str:
    text_n = normalize_text(text)
    words = re.findall(r"\b[\wÀ-ÿ']+\b", text_n)

    en_score = sum(1 for w in words if w in EN_HINTS)
    fr_score = sum(1 for w in words if w in FR_HINTS)

    if re.search(r"[àâçéèêëîïôùûü]", text_n):
        fr_score += 2

    return "en" if en_score > fr_score else "fr"

def has_prototype_phrase(text: str, forbidden: List[str]) -> bool:
    text_n = normalize_text(text)
    return any(normalize_text(p) in text_n for p in forbidden)

def keyword_hit_rate(text: str, expected_keywords: List[str]) -> float:
    text_n = normalize_text(text)
    if not expected_keywords:
        return 1.0
    hits = sum(1 for kw in expected_keywords if normalize_text(kw) in text_n)
    return hits / len(expected_keywords)

def latency_score(seconds: float) -> float:
    # 0 sec = 1.0 | 10 sec ou plus = 0.0
    return max(0.0, 1.0 - (seconds / 10.0))

def compute_auto_score(
    answer: str,
    latency: float,
    expected_lang: str,
    must_include_any: List[str],
    must_not_include: List[str],
) -> Dict[str, float]:
    lang_ok = 1.0 if detect_lang_simple(answer) == expected_lang else 0.0
    proto_ok = 0.0 if has_prototype_phrase(answer, must_not_include) else 1.0
    keyword_score = keyword_hit_rate(answer, must_include_any)
    speed_score = latency_score(latency)

    overall = (
        0.35 * keyword_score +
        0.25 * lang_ok +
        0.25 * proto_ok +
        0.15 * speed_score
    ) * 100.0

    return {
        "keyword_score": keyword_score,
        "lang_ok": lang_ok,
        "proto_ok": proto_ok,
        "speed_score": speed_score,
        "auto_score_100": overall,
    }

## Appel de l’API CityTaste

In [44]:
def ask_citytaste(question: str, context: Dict[str, Any] = None) -> Dict[str, Any]:
    payload = {
        "message": question,
        "context": context
    }

    start = time.perf_counter()
    r = requests.post(API_URL, json=payload, timeout=REQUEST_TIMEOUT)
    elapsed = time.perf_counter() - start

    r.raise_for_status()
    data = r.json()

    answer = data.get("answer", "")
    return {
        "raw": data,
        "answer": answer,
        "latency_sec": elapsed,
    }

## Exécution du benchmark

In [45]:
rows = []

for case in TEST_CASES:
    result = ask_citytaste(case["question"], context=case["context"])
    answer = result["answer"]
    latency = result["latency_sec"]

    scores = compute_auto_score(
        answer=answer,
        latency=latency,
        expected_lang=case["expected_lang"],
        must_include_any=case["must_include_any"],
        must_not_include=case["must_not_include"],
    )

    row = {
        "model": MODEL_NAME,
        "id": case["id"],
        "question": case["question"],
        "expected_lang": case["expected_lang"],
        "answer": answer,
        "latency_sec": latency,
        **scores,
        "manual_score_5": None,
        "notes": "",
    }
    rows.append(row)

df = pd.DataFrame(rows)
df

,model,id,question,expected_lang,answer,latency_sec,keyword_score,lang_ok,proto_ok,speed_score,auto_score_100,manual_score_5,notes
0,qwen3_4b,1,Trouve-moi un restaurant italien proche du cen...,fr,Voici quelques options :\n1. Paninaro — sandwi...,147.825743,0.25,1.0,1.0,0.000000,58.750000,None,
1,qwen3_4b,2,Je cherche un restaurant chinois près de St-La...,fr,Je n’ai pas trouvé de correspondance exacte po...,2.386031,0.60,1.0,1.0,0.761397,82.420953,None,
2,qwen3_4b,3,Je veux un hôtel près du centre-ville.,fr,Voici quelques options :\n1. Sheraton Ottawa H...,119.082143,0.50,0.0,1.0,0.000000,42.500000,None,
3,qwen3_4b,4,Find an Italian restaurant near downtown Ottawa.,en,Here are a few options:\n1. Paninaro — sandwic...,128.156605,0.25,0.0,1.0,0.000000,33.750000,None,
4,qwen3_4b,5,Find a hotel near downtown.,en,Here are a few options:\n1. Sheraton Ottawa Ho...,139.949129,0.50,1.0,1.0,0.000000,67.500000,None,
5,qwen3_4b,6,How do I use filters in CityTaste?,en,"Selon les données disponibles, CityTaste peut ...",9.200002,0.50,0.0,1.0,0.080000,43.699997,None,
6,qwen3_4b,7,Comment utiliser les filtres dans CityTaste ?,fr,"Selon les données disponibles, CityTaste peut ...",2.331298,0.50,1.0,1.0,0.766870,79.003053,None,
7,qwen3_4b,8,CityTaste permet-il de faire une réservation ?,fr,Non. CityTaste n’effectue pas de réservation à...,2.278196,0.75,1.0,1.0,0.772180,87.832706,None,
8,qwen3_4b,9,How are the results ranked?,en,Les résultats sont généralement ordonnés selon...,2.232528,0.25,0.0,1.0,0.776747,45.401208,None,
9,qwen3_4b,10,Comment activer ma position ?,fr,"Pour activer ta position, utilise l’option de ...",2.166532,1.00,1.0,1.0,0.783347,96.750202,None,


## Résumé automatique du modèle

In [23]:
summary = {
    "model": MODEL_NAME,
    "avg_latency_sec": df["latency_sec"].mean(),
    "avg_keyword_score": df["keyword_score"].mean(),
    "language_match_rate": df["lang_ok"].mean(),
    "no_prototype_rate": df["proto_ok"].mean(),
    "avg_auto_score_100": df["auto_score_100"].mean(),
}

summary_df = pd.DataFrame([summary])
summary_df

,model,avg_latency_sec,avg_keyword_score,language_match_rate,no_prototype_rate,avg_auto_score_100
0,llma_cpp_current,14.354857,0.575,0.75,1.0,72.485212


In [39]:
summary = {
    "model": MODEL_NAME,
    "avg_latency_sec": df["latency_sec"].mean(),
    "avg_keyword_score": df["keyword_score"].mean(),
    "language_match_rate": df["lang_ok"].mean(),
    "no_prototype_rate": df["proto_ok"].mean(),
    "avg_auto_score_100": df["auto_score_100"].mean(),
}

summary_df = pd.DataFrame([summary])
summary_df

,model,avg_latency_sec,avg_keyword_score,language_match_rate,no_prototype_rate,avg_auto_score_100
0,gemma3_4b,19.205821,0.7625,0.833333,1.0,78.886124


In [46]:
summary = {
    "model": MODEL_NAME,
    "avg_latency_sec": df["latency_sec"].mean(),
    "avg_keyword_score": df["keyword_score"].mean(),
    "language_match_rate": df["lang_ok"].mean(),
    "no_prototype_rate": df["proto_ok"].mean(),
    "avg_auto_score_100": df["auto_score_100"].mean(),
}

summary_df = pd.DataFrame([summary])
summary_df

,model,avg_latency_sec,avg_keyword_score,language_match_rate,no_prototype_rate,avg_auto_score_100
0,qwen3_4b,51.86111,0.591667,0.666667,1.0,68.253032



## Évaluation manuelle

Après la lecture des réponses, remplis la colonne `manual_score_5` :
- **5** = très bonne réponse
- **4** = bonne
- **3** = acceptable
- **2** = faible
- **1** = mauvaise

Puis ajoute éventuellement une note dans `notes`.



## Sauvegarde des résultats

Relancer ce notebook pour chaque modèle, en changeant simplement `MODEL_NAME`.


In [47]:
df.to_csv(f"benchmark_{MODEL_NAME}.csv", index=False)
summary_df.to_csv(f"benchmark_summary_{MODEL_NAME}.csv", index=False)
print("Résultats sauvegardés.")

Résultats sauvegardés.



## Comparer plusieurs modèles

Après avoir généré plusieurs fichiers CSV, mets leurs noms dans la liste ci-dessous.


In [49]:
from pathlib import Path
import pandas as pd

FILES_TO_COMPARE = [
    "benchmark_summary_llma_cpp_current.csv",
    "benchmark_summary_gemma3_4b.csv",
    "benchmark_summary_qwen3_4b.csv",
]

comparison = pd.concat([pd.read_csv(path) for path in FILES_TO_COMPARE], ignore_index=True)
comparison = comparison.sort_values(by="avg_auto_score_100", ascending=False)

comparison

,model,avg_latency_sec,avg_keyword_score,language_match_rate,no_prototype_rate,avg_auto_score_100
1,gemma3_4b,19.205821,0.762500,0.833333,1.0,78.886124
0,llma_cpp_current,14.354857,0.575000,0.750000,1.0,72.485212
2,qwen3_4b,51.861110,0.591667,0.666667,1.0,68.253032



## Conseils d’utilisation

Pour chaque modèle :
1. Lance le backend CityTaste avec le modèle voulu.
2. Mets à jour `MODEL_NAME`.
3. Exécute tout le notebook.
4. Sauvegarde les résultats.
5. Passe au modèle suivant.

### Exemple de noms
- `llama_cpp_current`
- `gemma3_4b`
- `qwen3_4b`

### Décision finale
Choisis le modèle qui combine :
- bonne moyenne automatique,
- bonne note manuelle,
- faible taux de phrases prototype,
- bonne langue,
- latence acceptable pour la démo.
